# Aula 08 — Testes para Duas Amostras e Qui-quadrado

## Objetivos
1. Comparar médias de **duas populações independentes** (t de Welch).
2. Comparar médias **pareadas** (antes/depois).
3. Testar **independência** entre variáveis categóricas (χ²).
4. Testar **aderência** a uma distribuição esperada (χ² de bondade).

---

## 1. Teste t para duas amostras independentes (Welch)

$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\dfrac{s_1^2}{n_1} + \dfrac{s_2^2}{n_2}}}$$

Não assume variâncias iguais — é a recomendação atual (em vez do Student clássico).

- $H_0$: $\mu_1 = \mu_2$
- $H_1$: $\mu_1 \ne \mu_2$ (ou unilateral)

## 2. Teste t pareado

Quando cada observação tem um "par" natural (mesmo indivíduo em dois momentos,
ou mesma UF em dois períodos). Trabalhamos com as **diferenças** $d_i = x_{1i} - x_{2i}$:

$$t = \frac{\bar{d}}{s_d/\sqrt{n}} \sim t_{n-1}$$

## 3. Teste qui-quadrado de independência

Testa se duas variáveis categóricas são **independentes** numa tabela de contingência.

$$\chi^2 = \sum_{ij} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}, \quad gl = (r-1)(c-1)$$

- $O_{ij}$ = frequência observada na célula $(i,j)$.
- $E_{ij}$ = frequência esperada sob independência = $\dfrac{\text{total linha} \cdot \text{total coluna}}{N}$.

## 4. Teste qui-quadrado de aderência

Testa se uma amostra segue uma **distribuição esperada**.

$$\chi^2 = \sum_k \frac{(O_k - E_k)^2}{E_k}, \quad gl = k - 1$$

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style="whitegrid")

from utils.sidra_api import obter_ipca_mensal, obter_pib_per_capita_uf

## 5. Aplicação 1: IPCA antes vs depois da pandemia

Vamos comparar o IPCA mensal em dois períodos:
- **Antes:** 2015-01 a 2019-12
- **Depois:** 2020-01 a 2024-12

- $H_0$: $\mu_{\text{antes}} = \mu_{\text{depois}}$
- $H_1$: $\mu_{\text{antes}} \ne \mu_{\text{depois}}$

In [ ]:
df = obter_ipca_mensal(120)
antes  = df[(df["data"] >= "2015-01-01") & (df["data"] <= "2019-12-31")]["variacao_mensal"].values
depois = df[(df["data"] >= "2020-01-01") & (df["data"] <= "2024-12-31")]["variacao_mensal"].values

print(f"Antes  — n={len(antes)},  média={antes.mean():.4f},  s={antes.std(ddof=1):.4f}")
print(f"Depois — n={len(depois)}, média={depois.mean():.4f}, s={depois.std(ddof=1):.4f}")

In [ ]:
# Welch's t-test (equal_var=False)
t_stat, p_valor = stats.ttest_ind(antes, depois, equal_var=False)
print(f"t = {t_stat:.4f}, p-valor = {p_valor:.4f}")
print("\nDecisão:", "REJEITA H0 — médias diferentes" if p_valor < 0.05 else "Não rejeita H0")

### Visualização da comparação

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot lado a lado
dfvis = pd.DataFrame({
    "valor": np.concatenate([antes, depois]),
    "periodo": ["Antes (2015-19)"]*len(antes) + ["Depois (2020-24)"]*len(depois)
})
sns.boxplot(data=dfvis, x="periodo", y="valor", ax=axes[0], palette="Set2")
axes[0].set_title("IPCA: distribuição antes vs depois")
axes[0].set_ylabel("Variação mensal (%)")

# KDE sobreposta
sns.kdeplot(antes,  ax=axes[1], label="Antes",  color="steelblue", linewidth=2)
sns.kdeplot(depois, ax=axes[1], label="Depois", color="darkorange", linewidth=2)
axes[1].set_title("Densidades sobrepostas")
axes[1].set_xlabel("IPCA mensal (%)")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula08_antes_depois.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Aplicação 2: teste pareado — PIB per capita em dois anos

Compararemos o PIB per capita das 27 UFs em dois anos. Como cada UF aparece nos dois
períodos, o **teste pareado** é o correto.

In [ ]:
df_a = obter_pib_per_capita_uf(ano=2019)[["uf", "pib_per_capita"]].rename(columns={"pib_per_capita": "pib_2019"})
df_b = obter_pib_per_capita_uf(ano=2021)[["uf", "pib_per_capita"]].rename(columns={"pib_per_capita": "pib_2021"})
pareado = df_a.merge(df_b, on="uf")
pareado["diferenca"] = pareado["pib_2021"] - pareado["pib_2019"]
pareado.head()

In [ ]:
# Teste t pareado
t_p, p_p = stats.ttest_rel(pareado["pib_2021"], pareado["pib_2019"])
print(f"Média da diferença: R$ {pareado['diferenca'].mean():,.2f}")
print(f"t pareado = {t_p:.4f}, p-valor = {p_p:.4f}")

if p_p < 0.05:
    print("\nREJEITA H0: há diferença estatisticamente significativa entre 2019 e 2021.")
else:
    print("\nNão rejeita H0.")

## 7. Aplicação 3: qui-quadrado de independência

Vamos categorizar UFs em "PIB Alto/Baixo" e "Região Sul/Sudeste vs Outras" e testar
se há associação.

In [ ]:
df_pib = obter_pib_per_capita_uf()

# Mapeamento simplificado de UF → grande região
sul_sudeste = {"São Paulo", "Rio de Janeiro", "Minas Gerais", "Espírito Santo",
               "Paraná", "Santa Catarina", "Rio Grande do Sul"}
df_pib["regiao"] = df_pib["uf"].apply(lambda u: "Sul/Sudeste" if u in sul_sudeste else "Outras")
df_pib["nivel_pib"] = pd.cut(df_pib["pib_per_capita"],
                              bins=[0, df_pib["pib_per_capita"].median(), np.inf],
                              labels=["Baixo", "Alto"])

# Tabela de contingência
tabela = pd.crosstab(df_pib["regiao"], df_pib["nivel_pib"])
print("Tabela observada:")
print(tabela)

In [ ]:
chi2, p_chi, gl, esperado = stats.chi2_contingency(tabela)

print(f"\nχ² = {chi2:.4f}")
print(f"gl = {gl}")
print(f"p-valor = {p_chi:.4f}")
print(f"\nFrequências esperadas (sob independência):")
print(pd.DataFrame(esperado, index=tabela.index, columns=tabela.columns).round(2))

if p_chi < 0.05:
    print("\nREJEITA H0: há associação entre região e nível de PIB.")
else:
    print("\nNão rejeita H0.")

### Heatmap dos resíduos padronizados

Resíduos padronizados grandes (em valor absoluto) apontam as células que mais
contribuem para o $\chi^2$.

In [ ]:
resid = (tabela - esperado) / np.sqrt(esperado)
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(resid, annot=True, cmap="RdBu_r", center=0, fmt=".2f", ax=ax,
            cbar_kws={"label": "Resíduo padronizado"})
ax.set_title("Resíduos do teste χ² — Região × Nível de PIB", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula08_chi2_residuos.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Qui-quadrado de aderência

Os meses com inflação alta (IPCA > 0,5%) estão **uniformemente distribuídos** pelos
trimestres? $H_0$: 25% em cada trimestre.

In [ ]:
df = obter_ipca_mensal(120)
altos = df[df["variacao_mensal"] > 0.5].copy()
altos["trimestre"] = altos["data"].dt.quarter
observado = altos["trimestre"].value_counts().sort_index()
esperado = np.full(4, observado.sum() / 4)

print("Observado por trimestre:")
print(observado.values)
print(f"Esperado (uniforme): {esperado}")

chi2_b, p_b = stats.chisquare(observado.values, f_exp=esperado)
print(f"\nχ² = {chi2_b:.4f}, p = {p_b:.4f}")
print("Resultado:", "REJEITA uniformidade" if p_b < 0.05 else "Compatível com uniformidade")

## Exercícios

1. Compare o IPCA dos semestres 1 e 2 (sazonalidade) com Welch.
2. Faça teste pareado para PIB per capita entre 2018 e 2020 — a pandemia teve efeito?
3. Crie uma categoria "Norte/Nordeste" vs "Outras" e refaça o χ² de independência.

**Próxima aula:** Correlação e regressão linear simples.